# Yaratık isimleri — Qwen3 & Gemma

Her üç model de `data/temiz_yaratik_isimleri.txt` üzerinde harf harf eğitildi. `load_checkpoint()` aynı kernel içinde mimari değiştirmeyi sağlar.


## Qwen3


In [2]:
import sys, importlib, torch

ARCH_MODULES = ("model", "config", "tokenizer", "block", "attention", "mlp", "rms_norm", "rotary")
ARCH_FOLDERS = ("qwen3", "gemma4", "qwen3_5", "deepseek3")


def load_checkpoint(folder: str, checkpoint: str, model_class: str):
    for name in list(sys.modules):
        if name in ARCH_MODULES:
            del sys.modules[name]
    for p in ARCH_FOLDERS:
        while p in sys.path:
            sys.path.remove(p)
    sys.path.insert(0, folder)

    model_mod = importlib.import_module("model")
    tok_mod = importlib.import_module("tokenizer")
    Model = getattr(model_mod, model_class)
    CharTokenizer = tok_mod.CharTokenizer

    ckpt = torch.load(checkpoint, map_location="cpu", weights_only=False)
    tok = CharTokenizer(ckpt["chars"])
    model = Model(ckpt["cfg"])
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, tok


model, tok = load_checkpoint("qwen3", "qwen3/tiny_qwen.pt", "TinyQwen")


In [3]:
model


TinyQwen(
  (embed_tokens): Embedding(27, 32)
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=16, bias=False)
        (v_proj): Linear(in_features=32, out_features=16, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=32, out_features=27, bias=False)
)

In [4]:
tok.encode("a")


[1]

In [5]:
tok.decode([0])


'\n'

In [6]:
model.embed_tokens.weight


Parameter containing:
tensor([[ 6.5720e-02, -9.0476e-02, -3.1367e-01, -7.4405e-01,  4.8080e-01,
         -5.0320e-02,  8.3218e-01,  2.8038e-02,  2.8132e-01,  1.0083e+00,
         -1.1636e+00, -4.7296e-01,  2.3184e-01, -2.4677e-01, -8.1384e-01,
          1.3407e+00,  1.2417e+00, -1.4716e-01,  3.1022e-01,  7.8069e-01,
         -1.8094e+00,  4.1702e-01,  1.3414e+00,  5.7346e-01,  9.9830e-02,
         -1.3285e+00, -9.8017e-01, -2.8530e-01,  3.7272e-01, -7.1907e-01,
          1.3369e+00,  2.2852e+00],
        [-5.6925e-01, -2.3573e-01,  7.9069e-01,  1.3996e-01,  1.2344e-01,
          8.1580e-01, -1.0033e+00, -2.5339e-01, -4.8232e-01, -7.3997e-01,
          4.8990e-01, -1.2528e+00, -9.8686e-01,  5.2368e-01, -4.8408e-01,
         -5.2831e-01,  1.4338e+00, -7.2040e-01,  1.1988e+00, -2.0269e-01,
         -1.2926e+00,  1.8638e+00,  2.4823e+00, -1.5855e+00,  1.2228e+00,
         -1.3406e+00,  7.3679e-01, -7.1980e-02,  6.4856e-01,  1.4527e+00,
          1.4108e+00, -2.8582e-01],
        [-6.5439e-

In [7]:
model.embed_tokens.weight[1]


tensor([-0.5693, -0.2357,  0.7907,  0.1400,  0.1234,  0.8158, -1.0033, -0.2534,
        -0.4823, -0.7400,  0.4899, -1.2528, -0.9869,  0.5237, -0.4841, -0.5283,
         1.4338, -0.7204,  1.1988, -0.2027, -1.2926,  1.8638,  2.4823, -1.5855,
         1.2228, -1.3406,  0.7368, -0.0720,  0.6486,  1.4527,  1.4108, -0.2858],
       grad_fn=<SelectBackward0>)

In [8]:
start = torch.full((10, 1), tok.eos_id, dtype=torch.long)   # every name starts at EOS/newline
out = model.generate(start, max_new_tokens=model.cfg.max_seq_len,
                     temperature=0.8, eos_id=tok.eos_id)     # stops each row at EOS
out


tensor([[ 0, 19,  4, 10,  4,  8,  3,  4, 15,  0,  0,  0,  0],
        [ 0,  9, 13, 15,  2, 22, 15, 23,  0,  0,  0,  0,  0],
        [ 0,  9, 13, 15, 14,  1, 15, 16,  0,  0,  0,  0,  0],
        [ 0, 19,  4, 10,  2, 22, 15, 23,  0,  0,  0,  0,  0],
        [ 0,  3,  4, 11,  7, 15,  2, 13, 19, 12, 18, 20,  0],
        [ 0,  1, 19,  2, 22, 15, 23,  0,  0,  0,  0,  0,  0],
        [ 0,  1,  9, 14,  4, 12, 21,  4,  0,  0,  0,  0,  0],
        [ 0,  3,  4, 11,  7, 15,  5, 22, 20,  0,  0,  0,  0],
        [ 0,  1,  9,  2, 13, 19, 12, 18, 20,  0,  0,  0,  0],
        [ 0,  9, 13, 15, 14,  4, 12, 21,  4,  0,  0,  0,  0]])

In [9]:
for row in out.tolist():
    print(tok.decode(row[1:]).split("\n")[0])


yelejder
korbörü
korpars
yelbörü
demirboynuz
aybörü
akpençe
demirgöz
akboynuz
korpençe


## Gemma

Aynı yaratık isimleri verisiyle eğitilmiş Gemma checkpoint'ini yükle ve karşılaştır.


In [10]:
gemma_model, gemma_tok = load_checkpoint("gemma4", "gemma4/tiny_gemma.pt", "TinyGemma")
gemma_model


TinyGemma(
  (embed_tokens): Embedding(27, 32)
  (per_layer_embeddings): Embedding(27, 192)
  (layers): ModuleList(
    (0-5): 6 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=16, bias=False)
        (v_proj): Linear(in_features=32, out_features=16, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (post_attention_layernorm): RMSNorm()
      (pre_feedforward_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
      (post_feedforward_layernorm): RMSNorm()
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=32, out

In [11]:
start = torch.full((10, 1), gemma_tok.eos_id, dtype=torch.long)
gemma_out = gemma_model.generate(
    start,
    max_new_tokens=gemma_model.cfg.max_seq_len,
    temperature=0.8,
    eos_id=gemma_tok.eos_id,
)
gemma_out


tensor([[ 0,  3,  4, 11,  7, 15,  9,  1, 12,  1, 17,  0],
        [ 0,  5, 22,  9,  4, 15,  0,  0,  0,  0,  0,  0],
        [ 0,  1,  9,  2, 22, 15, 23,  0,  0,  0,  0,  0],
        [ 0,  1, 17,  4, 26,  5, 22, 20,  0,  0,  0,  0],
        [ 0,  2, 13, 20,  2, 22, 15, 23,  0,  0,  0,  0],
        [ 0, 19,  4, 10,  2, 22, 15, 23,  0,  0,  0,  0],
        [ 0, 19,  4, 10,  9, 18, 15, 17,  0,  0,  0,  0],
        [ 0,  9,  1, 15,  1,  2, 22, 15, 23,  0,  0,  0],
        [ 0,  3,  4, 11,  7, 15,  5, 22, 20,  0,  0,  0],
        [ 0,  5, 22,  9,  2, 22, 15, 23,  0,  0,  0,  0]])

In [12]:
for row in gemma_out.tolist():
    print(gemma_tok.decode(row[1:]).split("\n")[0])


demirkanat
göker
akbörü
ateşgöz
bozbörü
yelbörü
yelkurt
karabörü
demirgöz
gökbörü


## Deepseek3

Aynı yaratık isimleri verisiyle eğitilmiş Deepseek checkpoint'ini yükle ve karşılaştır.


In [13]:
deepseek_model, deepseek_tok = load_checkpoint("deepseek3", "deepseek3/tiny_deepseek.pt", "TinyDeepSeek")
start = torch.full((10, 1), deepseek_tok.eos_id, dtype=torch.long)
deepseek_out = deepseek_model.generate(
    start,
    max_new_tokens=deepseek_model.cfg.max_seq_len,
    temperature=0.8,
    eos_id=deepseek_tok.eos_id,
)
for row in deepseek_out.tolist():
    print(deepseek_tok.decode(row[1:]).split("\n")[0])



ateşalp
akkanat
demirbörü
sisboynuz
sisboynuz
yelhan
ateşkurt
yelgöz
demirkurt
alphhan
